In [ ]:
import secrets
from fastapi.testclient import TestClient
from radar import store
from radar.app import app

client = TestClient(app)
print('initial approved', store.row('SELECT COUNT(*) AS c FROM news WHERE approved=1'))
for path in ['/', '/api/health', '/api/news', '/api/sources', '/api/config', '/api/boletin.html', '/api/boletin.png']:
    r = client.get(path)
    print(path, r.status_code)
print('after public approved', store.row('SELECT COUNT(*) AS c FROM news WHERE approved=1'))
email = f"qa-{secrets.token_hex(5)}@example.com"
data = {'email': email, 'password': 'TestPassword_123', 'terms_accepted': True, 'privacy_acknowledged': True}
print('invalid register', client.post('/api/auth/register', json={**data, 'terms_accepted': False}).status_code)
r = client.post('/api/auth/register', json=data)
print('valid register', r.status_code, r.text)
print('me', client.get('/api/auth/me').json())
resp = client.get('/api/news')
print('news status', resp.status_code)
print('news items', len(resp.json().get('items', [])))
print('sample rows', store.rows('SELECT id, slug, headline, approved, is_demo FROM news ORDER BY id DESC LIMIT 10'))

initial approved {'c': 7}
/ 200
/api/health 200
RADAR: ronda RSS completada -> 300 consultadas, 0 nuevas, 0 actualizadas, errores=17
/api/news 200
/api/sources 200
/api/config 200
/api/boletin.html 200
/api/boletin.png 200
after public approved {'c': 7}
invalid register 422
valid register 200 {"id":42,"email":"qa-61b72a399c@example.com","role":"reader"}
me {'user': {'id': 42, 'email': 'qa-61b72a399c@example.com', 'role': 'reader', 'created_at': '2026-11-11T20:38:00+00:00'}}
news status 200
news items 7
sample rows [{'id': 149, 'slug': 'n-032e2306c7440e', 'headline': 'Fuerza Aérea de Colombia revela la razón de los sobrevuelos de aviones de guerra sobre Bogotá este miércoles - NTN24', 'approved': 0, 'is_demo': 0}, {'id': 148, 'slug': 'n-684cfa3ff72101', 'headline': '¡Atención, beneficiarios! Más de 4000 personas no han cobrado el subsidio Colombia Mayor en Cali - Alcaldía de Santiago de Cali', 'approved': 0, 'is_demo': 0}, {'id': 147, 'slug': 'n-df93ff137f255a', 'headline': "We're addin

: 